# CNN clásicas para clasificación de lesiones dermatológicas con HAM10000

Versión adaptada para comparación científica homogénea con modelos modernos como EfficientNet-B0, ConvNeXt-Tiny y ViT-B/16.

Arquitecturas evaluadas:

- ResNet50
- DenseNet121
- InceptionV3

Mejoras metodológicas principales:

- Split estricto por `lesion_id` para evitar *data leakage*.
- Separación train / validation / test a nivel de lesión.
- Sin duplicación física de imágenes.
- Balanceo mediante `WeightedRandomSampler`.
- Focal Loss ponderada por clase.
- Augmentación solo en entrenamiento.
- Métricas robustas: accuracy, balanced accuracy, macro F1, weighted F1, ROC-AUC OvR y matriz de confusión.
- Checkpoint del mejor modelo según `macro_f1` de validación.
- Protocolo homogéneo para comparación científica.

In [1]:
# ============================================================
# 1. INSTALACIÓN / IMPORTS
# ============================================================

!pip -q install kagglehub scikit-learn seaborn

import os
import random
from pathlib import Path
from collections import Counter

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# ============================================================
# 2. SEMILLA Y DISPOSITIVO
# ============================================================

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Dispositivo: cuda
GPU: NVIDIA A100-SXM4-80GB


In [3]:
# ============================================================
# 3. DESCARGA Y CARGA DE HAM10000
# ============================================================

print("=" * 80)
print("DESCARGANDO HAM10000")
print("=" * 80)

data_dir = Path(kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000"))
print("Dataset descargado en:", data_dir)

all_image_paths = list(data_dir.glob("ham10000_images_part_1/*.jpg")) + list(data_dir.glob("ham10000_images_part_2/*.jpg"))
imageid_path_dict = {p.stem: str(p) for p in all_image_paths}

lesion_type_dict = {
    "nv": "Melanocytic nevi",
    "mel": "Melanoma",
    "bkl": "Benign keratosis-like lesions",
    "bcc": "Basal cell carcinoma",
    "akiec": "Actinic keratoses",
    "vasc": "Vascular lesions",
    "df": "Dermatofibroma",
}

metadata_path = data_dir / "HAM10000_metadata.csv"
df = pd.read_csv(metadata_path)

df["path"] = df["image_id"].map(imageid_path_dict)
df["cell_type"] = df["dx"].map(lesion_type_dict)

CLASS_ORDER = ["akiec", "bcc", "bkl", "df", "nv", "vasc", "mel"]
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASS_ORDER)}
CLASS_TO_IDX = {c: i for i, c in IDX_TO_CLASS.items()}

df["label"] = df["dx"].map(CLASS_TO_IDX)

print("Número total de muestras:", len(df))
print("Número de lesion_id únicos:", df["lesion_id"].nunique())
print(df["dx"].value_counts())
df.head()

DESCARGANDO HAM10000
Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Dataset descargado en: /kaggle/input/skin-cancer-mnist-ham10000
Número total de muestras: 10015
Número de lesion_id únicos: 7470
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


,lesion_id,image_id,dx,dx_type,age,sex,localization,path,cell_type,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/kaggle/input/skin-cancer-mnist-ham10000/ham10...,Benign keratosis-like lesions,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/kaggle/input/skin-cancer-mnist-ham10000/ham10...,Benign keratosis-like lesions,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/kaggle/input/skin-cancer-mnist-ham10000/ham10...,Benign keratosis-like lesions,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/kaggle/input/skin-cancer-mnist-ham10000/ham10...,Benign keratosis-like lesions,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/kaggle/input/skin-cancer-mnist-ham10000/ham10...,Benign keratosis-like lesions,2


In [4]:
# ============================================================
# 4. SPLIT ESTRICTO POR LESION_ID
#    Evita fuga de información entre train / val / test
# ============================================================

lesion_df = (
    df.groupby("lesion_id")
      .agg(dx=("dx", "first"))
      .reset_index()
)

train_lesions, temp_lesions = train_test_split(
    lesion_df,
    test_size=0.30,
    random_state=SEED,
    stratify=lesion_df["dx"]
)

val_lesions, test_lesions = train_test_split(
    temp_lesions,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_lesions["dx"]
)

train_ids = set(train_lesions["lesion_id"])
val_ids = set(val_lesions["lesion_id"])
test_ids = set(test_lesions["lesion_id"])

train_df = df[df["lesion_id"].isin(train_ids)].copy().reset_index(drop=True)
val_df   = df[df["lesion_id"].isin(val_ids)].copy().reset_index(drop=True)
test_df  = df[df["lesion_id"].isin(test_ids)].copy().reset_index(drop=True)

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("Train imágenes:", len(train_df), "| lesion_id únicos:", train_df["lesion_id"].nunique())
print("Val imágenes:  ", len(val_df),   "| lesion_id únicos:", val_df["lesion_id"].nunique())
print("Test imágenes: ", len(test_df),  "| lesion_id únicos:", test_df["lesion_id"].nunique())

print("\nDistribución train:")
print(train_df["dx"].value_counts().sort_index())

print("\nDistribución val:")
print(val_df["dx"].value_counts().sort_index())

print("\nDistribución test:")
print(test_df["dx"].value_counts().sort_index())

Train imágenes: 6981 | lesion_id únicos: 5229
Val imágenes:   1532 | lesion_id únicos: 1120
Test imágenes:  1502 | lesion_id únicos: 1121

Distribución train:
dx
akiec     222
bcc       361
bkl       772
df         71
mel       773
nv       4683
vasc       99
Name: count, dtype: int64

Distribución val:
dx
akiec      53
bcc        82
bkl       160
df         24
mel       173
nv       1018
vasc       22
Name: count, dtype: int64

Distribución test:
dx
akiec      52
bcc        71
bkl       167
df         20
mel       167
nv       1004
vasc       21
Name: count, dtype: int64


In [5]:
# ============================================================
# 5. NORMALIZACIÓN Y TRANSFORMACIONES
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [6]:
# ============================================================
# 6. DATASET
# ============================================================

class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = HAM10000Dataset(train_df, transform=train_transform)
val_dataset   = HAM10000Dataset(val_df, transform=eval_transform)
test_dataset  = HAM10000Dataset(test_df, transform=eval_transform)

In [7]:
# ============================================================
# 7. BALANCEO MEDIANTE WeightedRandomSampler
# ============================================================

train_labels = train_df["label"].values

class_sample_count = np.array([
    (train_labels == i).sum()
    for i in range(len(CLASS_ORDER))
])

class_weights_sampler = 1.0 / class_sample_count
sample_weights = class_weights_sampler[train_labels]
sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Conteo por clase en train:", class_sample_count)
print("Pesos sampler por clase:", class_weights_sampler)

Conteo por clase en train: [ 222  361  772   71 4683   99  773]
Pesos sampler por clase: [0.0045045  0.00277008 0.00129534 0.01408451 0.00021354 0.01010101
 0.00129366]


In [8]:
# ============================================================
# 8. FOCAL LOSS PONDERADA
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits,
            targets,
            weight=self.alpha,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_ORDER)),
    y=train_labels
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

criterion = FocalLoss(
    alpha=class_weights_tensor,
    gamma=2.0
)

print("Pesos de clase para Focal Loss:", class_weights)

Pesos de clase para Focal Loss: [ 4.49227799  2.76256431  1.29182087 14.04627767  0.21295873 10.07359307
  1.2901497 ]


In [9]:
# ============================================================
# 9. CONSTRUCCIÓN DE MODELOS CNN CLÁSICOS
# ============================================================

NUM_CLASSES = len(CLASS_ORDER)

def build_resnet50(num_classes=NUM_CLASSES):
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model

def build_densenet121(num_classes=NUM_CLASSES):
    weights = models.DenseNet121_Weights.IMAGENET1K_V1
    model = models.densenet121(weights=weights)

    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)

    return model

def build_inception_v3(num_classes=NUM_CLASSES):
    weights = models.Inception_V3_Weights.IMAGENET1K_V1
    model = models.inception_v3(weights=weights, aux_logits=True)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    if model.AuxLogits is not None:
        aux_in_features = model.AuxLogits.fc.in_features
        model.AuxLogits.fc = nn.Linear(aux_in_features, num_classes)

    return model

models_dict = {
    "ResNet50": build_resnet50,
    "DenseNet121": build_densenet121,
    "InceptionV3": build_inception_v3,
}

In [10]:
# ============================================================
# 10. FUNCIONES DE ENTRENAMIENTO Y EVALUACIÓN
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, device, model_name=""):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_targets = []

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        # InceptionV3 devuelve InceptionOutputs durante entrenamiento
        if hasattr(outputs, "logits"):
            logits = outputs.logits
            aux_logits = outputs.aux_logits
            loss_main = criterion(logits, targets)
            loss_aux = criterion(aux_logits, targets) if aux_logits is not None else 0
            loss = loss_main + 0.4 * loss_aux
        else:
            logits = outputs
            loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_bal_acc = balanced_accuracy_score(all_targets, all_preds)
    epoch_macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)

    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "balanced_accuracy": epoch_bal_acc,
        "macro_f1": epoch_macro_f1
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_logits = []
    all_probs = []
    all_preds = []
    all_targets = []

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        outputs = model(images)

        if hasattr(outputs, "logits"):
            logits = outputs.logits
        else:
            logits = outputs

        loss = criterion(logits, targets)
        running_loss += loss.item() * images.size(0)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_logits.append(logits.detach().cpu())
        all_probs.append(probs.detach().cpu())
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())

    all_probs = torch.cat(all_probs).numpy()
    all_targets_np = np.array(all_targets)
    all_preds_np = np.array(all_preds)

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_targets_np, all_preds_np)
    bal_acc = balanced_accuracy_score(all_targets_np, all_preds_np)
    macro_f1 = f1_score(all_targets_np, all_preds_np, average="macro", zero_division=0)
    weighted_f1 = f1_score(all_targets_np, all_preds_np, average="weighted", zero_division=0)

    try:
        roc_auc = roc_auc_score(
            all_targets_np,
            all_probs,
            multi_class="ovr",
            average="macro"
        )
    except ValueError:
        roc_auc = np.nan

    return {
        "loss": epoch_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "roc_auc_ovr": roc_auc,
        "y_true": all_targets_np,
        "y_pred": all_preds_np,
        "y_prob": all_probs
    }

In [ ]:
# ============================================================
# 11. ENTRENAMIENTO COMPLETO CON CHECKPOINT POR MACRO F1
# ============================================================

EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5

results = {}
histories = {}

def train_model(model_name, build_fn):
    print("=" * 80)
    print(f"ENTRENANDO {model_name}")
    print("=" * 80)

    set_seed(SEED)

    model = build_fn().to(device)

    optimizer = AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    best_macro_f1 = -1
    best_path = f"best_{model_name}.pt"
    patience_counter = 0

    history = []

    for epoch in range(1, EPOCHS + 1):
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            model_name=model_name
        )

        val_metrics = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_roc_auc_ovr": val_metrics["roc_auc_ovr"],
        }

        history.append(row)

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={row['train_loss']:.4f} | "
            f"train_macro_f1={row['train_macro_f1']:.4f} | "
            f"val_loss={row['val_loss']:.4f} | "
            f"val_macro_f1={row['val_macro_f1']:.4f} | "
            f"val_bal_acc={row['val_balanced_accuracy']:.4f}"
        )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            torch.save(model.state_dict(), best_path)
            patience_counter = 0
            print(f"Nuevo mejor modelo guardado: {best_path}")
        else:
            patience_counter += 1
            print(f"Sin mejora. Paciencia: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print("Early stopping activado.")
            break

    model.load_state_dict(torch.load(best_path, map_location=device))

    test_metrics = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    results[model_name] = test_metrics
    histories[model_name] = pd.DataFrame(history)

    print("\nRESULTADOS TEST:", model_name)
    print({
        k: v for k, v in test_metrics.items()
        if k not in ["y_true", "y_pred", "y_prob"]
    })

    return model


trained_models = {}

for model_name, build_fn in models_dict.items():
    trained_models[model_name] = train_model(model_name, build_fn)

ENTRENANDO ResNet50
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 154MB/s]


Epoch 01/20 | train_loss=2.4352 | train_macro_f1=0.4355 | val_loss=0.9647 | val_macro_f1=0.2461 | val_bal_acc=0.5636
Nuevo mejor modelo guardado: best_ResNet50.pt
Epoch 02/20 | train_loss=0.6063 | train_macro_f1=0.6558 | val_loss=0.8279 | val_macro_f1=0.2694 | val_bal_acc=0.6051
Nuevo mejor modelo guardado: best_ResNet50.pt
Epoch 03/20 | train_loss=0.3578 | train_macro_f1=0.7109 | val_loss=0.8204 | val_macro_f1=0.3675 | val_bal_acc=0.6476
Nuevo mejor modelo guardado: best_ResNet50.pt
Epoch 04/20 | train_loss=0.3094 | train_macro_f1=0.7569 | val_loss=0.7341 | val_macro_f1=0.4119 | val_bal_acc=0.6951
Nuevo mejor modelo guardado: best_ResNet50.pt
Epoch 05/20 | train_loss=0.2482 | train_macro_f1=0.7899 | val_loss=0.7593 | val_macro_f1=0.4309 | val_bal_acc=0.6824
Nuevo mejor modelo guardado: best_ResNet50.pt
Epoch 06/20 | train_loss=0.2515 | train_macro_f1=0.7981 | val_loss=0.9321 | val_macro_f1=0.4614 | val_bal_acc=0.6618
Nuevo mejor modelo guardado: best_ResNet50.pt
Epoch 07/20 | train_lo

100%|██████████| 30.8M/30.8M [00:00<00:00, 228MB/s]


Epoch 01/20 | train_loss=1.7993 | train_macro_f1=0.5252 | val_loss=0.7923 | val_macro_f1=0.2701 | val_bal_acc=0.5936
Nuevo mejor modelo guardado: best_DenseNet121.pt
Epoch 02/20 | train_loss=0.5723 | train_macro_f1=0.6592 | val_loss=0.7977 | val_macro_f1=0.3357 | val_bal_acc=0.6421
Nuevo mejor modelo guardado: best_DenseNet121.pt
Epoch 03/20 | train_loss=0.3701 | train_macro_f1=0.7236 | val_loss=0.8148 | val_macro_f1=0.3612 | val_bal_acc=0.6291
Nuevo mejor modelo guardado: best_DenseNet121.pt
Epoch 04/20 | train_loss=0.2533 | train_macro_f1=0.7668 | val_loss=0.7739 | val_macro_f1=0.4209 | val_bal_acc=0.6359
Nuevo mejor modelo guardado: best_DenseNet121.pt
Epoch 05/20 | train_loss=0.2357 | train_macro_f1=0.7883 | val_loss=0.9036 | val_macro_f1=0.4233 | val_bal_acc=0.6314
Nuevo mejor modelo guardado: best_DenseNet121.pt
Epoch 06/20 | train_loss=0.1481 | train_macro_f1=0.8126 | val_loss=0.8288 | val_macro_f1=0.4308 | val_bal_acc=0.6305
Nuevo mejor modelo guardado: best_DenseNet121.pt
Epoc

In [ ]:
# ============================================================
# 12. TABLA RESUMEN DE RESULTADOS
# ============================================================

summary_rows = []

for model_name, metrics in results.items():
    summary_rows.append({
        "modelo": model_name,
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "roc_auc_ovr": metrics["roc_auc_ovr"],
        "loss": metrics["loss"],
    })

summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
summary_df

In [ ]:
# ============================================================
# 13. INFORMES DE CLASIFICACIÓN
# ============================================================

target_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

for model_name, metrics in results.items():
    print("=" * 80)
    print(model_name)
    print("=" * 80)

    print(classification_report(
        metrics["y_true"],
        metrics["y_pred"],
        target_names=target_names,
        zero_division=0
    ))

In [ ]:
# ============================================================
# 14. MATRICES DE CONFUSIÓN
# ============================================================

def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=target_names,
        yticklabels=target_names
    )
    plt.title(title)
    plt.xlabel("Predicción")
    plt.ylabel("Clase real")
    plt.tight_layout()
    plt.show()

for model_name, metrics in results.items():
    plot_confusion_matrix(
        metrics["y_true"],
        metrics["y_pred"],
        f"Matriz de confusión - {model_name}"
    )

In [ ]:
# ============================================================
# 15. CURVAS DE ENTRENAMIENTO
# ============================================================

for model_name, history_df in histories.items():
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_macro_f1"], label="Train Macro F1")
    plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="Val Macro F1")
    plt.title(f"Evolución Macro F1 - {model_name}")
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
    plt.title(f"Evolución Loss - {model_name}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 16. ANÁLISIS VISUAL DE ERRORES
# ============================================================

def show_misclassified(model, dataframe, title, n=12):
    model.eval()

    wrong_examples = []

    for idx in range(len(dataframe)):
        row = dataframe.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        input_tensor = eval_transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(input_tensor)
            if hasattr(outputs, "logits"):
                logits = outputs.logits
            else:
                logits = outputs

            prob = torch.softmax(logits, dim=1)
            pred = torch.argmax(prob, dim=1).item()

        true_label = int(row["label"])

        if pred != true_label:
            wrong_examples.append((row["path"], true_label, pred))

        if len(wrong_examples) >= n:
            break

    if not wrong_examples:
        print(f"No se encontraron errores para {title}")
        return

    cols = 4
    rows = int(np.ceil(len(wrong_examples) / cols))

    plt.figure(figsize=(16, rows * 4))
    plt.suptitle(f"Ejemplos mal clasificados - {title}", fontsize=16)

    for i, (path, true_label, pred_label) in enumerate(wrong_examples):
        image = Image.open(path).convert("RGB")

        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.axis("off")
        plt.title(
            f"Real: {IDX_TO_CLASS[true_label]}\nPred: {IDX_TO_CLASS[pred_label]}",
            fontsize=10
        )

    plt.tight_layout()
    plt.show()

for model_name, model in trained_models.items():
    show_misclassified(
        model,
        test_df,
        model_name,
        n=12
    )

In [ ]:
# ============================================================
# 17. EXPORTACIÓN DE RESULTADOS
# ============================================================

summary_df.to_csv("resultados_cnn_clasicas_ham10000.csv", index=False)

for model_name, history_df in histories.items():
    history_df.to_csv(f"historial_{model_name}.csv", index=False)

print("Archivos exportados:")
print("- resultados_cnn_clasicas_ham10000.csv")
for model_name in histories:
    print(f"- historial_{model_name}.csv")

# Nota metodológica para el paper

Todos los modelos CNN clásicos fueron entrenados bajo el mismo protocolo experimental empleado para las arquitecturas modernas. La partición del dataset se realizó a nivel de lesión (`lesion_id`) para evitar fuga de información entre entrenamiento, validación y prueba. El desbalance de clases se abordó mediante `WeightedRandomSampler` y Focal Loss ponderada. La selección del mejor modelo se basó en el valor de `macro_f1` sobre validación, priorizando el rendimiento equilibrado entre clases frente a la accuracy global.

Este protocolo permite una comparación más justa entre ResNet50, DenseNet121, InceptionV3, EfficientNet-B0, ConvNeXt-Tiny y ViT-B/16.